# Phase 7 — Chat Context Testing

Testing Claude with call + project file context in isolation before wiring the frontend.

**Steps:**
1. Setup — paths, imports, DB connection
2. Inventory — what calls and files are available
3. Load a single call into context (briefing + transcript)
4. Load project files into context
5. Assemble full system prompt — eyeball it
6. Count tokens before calling
7. First real call — single question, non-streaming
8. Multi-turn conversation
9. Multi-item context — 2 calls + a file
10. Streaming
11. Token budget stress test — all 12 calls

## 1. Setup

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()               # phase-7-chat/
ROOT_DIR     = NOTEBOOK_DIR.parent            # sundial_meetings/
BACKEND_DIR  = ROOT_DIR / 'backend'
NOTES_DIR    = ROOT_DIR / 'dummy_data' / 'notes'
RUNS_DIR     = BACKEND_DIR / 'runs'
DB_PATH      = BACKEND_DIR / 'jobs.db'
SERVICES_DIR = ROOT_DIR / 'services'

for p in (SERVICES_DIR, ROOT_DIR / 'utils'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import anthropic
from anthropic_client import call_with_retry  # our retry wrapper

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
MODEL  = 'claude-sonnet-4-6'

print(f'anthropic SDK: {anthropic.__version__}')
print(f'DB: {DB_PATH}')
print(f'Runs: {RUNS_DIR}')
print(f'Notes: {NOTES_DIR}')

anthropic SDK: 0.84.0
DB: /Users/anaolano/Desktop/CODE/SUNDIAL_MEETINGS/sundial_meetings/backend/jobs.db
Runs: /Users/anaolano/Desktop/CODE/SUNDIAL_MEETINGS/sundial_meetings/backend/runs
Notes: /Users/anaolano/Desktop/CODE/SUNDIAL_MEETINGS/sundial_meetings/dummy_data/notes


## 2. Inventory — what's available

In [2]:
# ── DB helpers ────────────────────────────────────────────────────────────────
def _connect():
    conn = sqlite3.connect(DB_PATH, isolation_level=None)
    conn.row_factory = sqlite3.Row
    return conn

def _find_run_dir(job_id: str) -> Path | None:
    """Read run_dir from DB"""
    conn = _connect()
    row  = conn.execute('SELECT run_dir FROM jobs WHERE id = ?', (job_id,)).fetchone()
    conn.close()
    if row and row['run_dir']:
        p = Path(row['run_dir'])
        return p if p.exists() else None
    return None

# ── List parent calls ─────────────────────────────────────────────────────────
conn  = _connect()
rows  = conn.execute(
    "SELECT id, transcript_name, created_at, status, briefing FROM jobs "
    "WHERE parent_job_id IS NULL ORDER BY created_at DESC"
).fetchall()
conn.close()

print(f'{len(rows)} parent calls:\n')
for r in rows:
    run_dir    = _find_run_dir(r['id'])
    has_tx     = '✓ tx' if run_dir and (run_dir / 'transcript.txt').exists() else '✗ tx'
    has_brief  = '✓ brief' if r['briefing'] else '✗ brief'
    print(f"  {r['id'][:8]}  {r['transcript_name']:<40} {r['status']:<8} {has_tx}  {has_brief}")

13 parent calls:

  da985b3b  260403_Sundial build checkin             done     ✓ tx  ✓ brief
  301a396d  260402_Jon Wendt                         done     ✓ tx  ✓ brief
  8d0d1ef6  260402_Justin Jankowski                  done     ✓ tx  ✓ brief
  6cf0f4c3  260402_Jodi Parmeter                     done     ✓ tx  ✓ brief
  dc60a454  260401_Jeff Parmeter                     done     ✓ tx  ✓ brief
  85562292  260401_Lukas Eichinger                   done     ✓ tx  ✓ brief
  4d188f07  260401_Shon Flaherty                     done     ✓ tx  ✓ brief
  fe967d3d  call-05-followup                         done     ✓ tx  ✓ brief
  0dfe2ff9  call-04-crm-frontend-design              done     ✓ tx  ✓ brief
  f9143128  call-03-sales-crm-followup               done     ✓ tx  ✓ brief
  2e8f11a9  call-02-transcription-demo               done     ✓ tx  ✓ brief
  9706bd1c  call-01-golden-eagle-discovery           done     ✓ tx  ✓ brief
  45caff60  26_04_02_Jodi Parmeter (fixed)           done     ✓ tx  ✓ 

In [3]:
# ── List project files ────────────────────────────────────────────────────────
md_files = sorted(NOTES_DIR.rglob('*.md'))
print(f'{len(md_files)} project files:\n')
for f in md_files:
    rel   = f.relative_to(NOTES_DIR)
    chars = len(f.read_text(encoding='utf-8'))
    print(f'  {str(rel):<40} {chars:>6} chars  (~{chars//4:>5} tokens est.)')

14 project files:

  other-notes.md                             9237 chars  (~ 2309 tokens est.)
  people/andy-eichinger.md                     36 chars  (~    9 tokens est.)
  people/chris-stitcher.md                    103 chars  (~   25 tokens est.)
  people/jay-eichinger.md                     813 chars  (~  203 tokens est.)
  people/jeff-parmeter.md                    6087 chars  (~ 1521 tokens est.)
  people/jodi-parmeter.md                    6847 chars  (~ 1711 tokens est.)
  people/john-wednt.md                       5659 chars  (~ 1414 tokens est.)
  people/justin-jankowski.md                 7026 chars  (~ 1756 tokens est.)
  people/lucas-eichinger.md                  3570 chars  (~  892 tokens est.)
  people/sean-flaherty.md                    3040 chars  (~  760 tokens est.)
  people/tammy.md                            1804 chars  (~  451 tokens est.)
  project-overview.md                       37710 chars  (~ 9427 tokens est.)
  sales-meetings.md                         2

## 3. Load a single call into context

In [ ]:
def load_call_context(job_id: str, include_transcript: bool = True) -> str:
    """
    Load one call from DB + run folder and format as a context block.
    Returns a string ready to be included in the system prompt.
    """
    conn = _connect()
    row  = conn.execute('SELECT * FROM jobs WHERE id = ?', (job_id,)).fetchone()
    conn.close()
    if not row:
        raise ValueError(f'Job {job_id} not found')

    name = row['transcript_name']
    date = row['created_at'][:10] if row['created_at'] else ''

    parts = [f'## Call: {name} ({date})']

    # Briefing sections (skip engagement_context — it's internal scaffolding)
    if row['briefing']:
        b = json.loads(row['briefing'])
        for section, label in [
            ('summary',      'Summary'),
            ('attendees',    'Attendees'),
            ('topics',       'Topics'),
            ('key_items',    'Key Items'),
            ('action_items', 'Action Items'),
            ('email_draft',  'Follow-up Email Draft'),
        ]:
            if b.get(section):
                parts.append(f'\n### {label}\n{b[section]}')

    # Full transcript
    if include_transcript:
        run_dir = _find_run_dir(job_id)
        tx_file = run_dir / 'transcript.txt' if run_dir else None
        if tx_file and tx_file.exists():
            tx = tx_file.read_text(encoding='utf-8')
            parts.append(f'\n### Full Transcript\n{tx}')
        else:
            parts.append('\n### Full Transcript\n(not available)')

    return '\n'.join(parts)


# ── Test: load the first available call ───────────────────────────────────────
# Change this to any id from the inventory above
TEST_JOB_ID = rows[0]['id']

ctx = load_call_context(TEST_JOB_ID)
print(f'Context block for: {rows[0]["transcript_name"]}')
print(f'Length: {len(ctx):,} chars  (~{len(ctx)//4:,} tokens estimated)\n')
print('─' * 60)
print(ctx[:2000])  # preview first 2000 chars
print('...' if len(ctx) > 2000 else '')

Context block for: 260403_Sundial build checkin
Length: 32,444 chars  (~8,111 tokens estimated)

────────────────────────────────────────────────────────────
## Call: 260403_Sundial build checkin (2026-04-03)

### Summary
This was an internal working session between Ana and Daniel. The call covered progress updates on their respective builds — Ana demonstrated her meeting intelligence dashboard (GUI, email template generation, and a pipeline that auto-updates Markdown knowledge files from call transcripts), and Daniel showed an annotation tool he built for collecting structured UI feedback. They discussed technical architecture questions including knowledge base design, Claude skill-based workflows, Obsidian as a comparator, and how to integrate the four communication channels (calls via Elevate, video via Teams, SMS, and email) into a unified client communication log. They also flagged on-prem data as a potential integration risk and agreed Ana should coordinate with Sean on data arch

## 4. Load project files into context

In [6]:
def load_file_context(rel: str) -> str:
    """
    Load one project file and format as a context block.
    `rel` is relative to NOTES_DIR, e.g. 'people/jay-eichinger.md'
    """
    path = NOTES_DIR / rel
    if not path.exists():
        raise FileNotFoundError(f'{rel} not found in {NOTES_DIR}')
    content = path.read_text(encoding='utf-8')
    return f'## Project File: {rel}\n\n{content}'


# ── Test: load a file ─────────────────────────────────────────────────────────
# Change to any rel path from the inventory above
TEST_FILE_REL = str(md_files[0].relative_to(NOTES_DIR)) if md_files else None

if TEST_FILE_REL:
    file_ctx = load_file_context(TEST_FILE_REL)
    print(f'File context for: {TEST_FILE_REL}')
    print(f'Length: {len(file_ctx):,} chars\n')
    print(file_ctx[:1500])
else:
    print('No .md files found in notes dir')

File context for: other-notes.md
Length: 9,270 chars

## Project File: other-notes.md

# Other Notes

## JumpFly (Vendor)
- External agency managing paid search and SEO for Golden Eagle.
- Paid search division: managing ~5,500 keywords on Google.
- SEO division: ~3 people working on organic search for ~1.5 years; provide content recommendations and SEO reports.
- Provide Google Analytics reporting to Sean and Jay.
- Also use phone number tracking for ad attribution.
- Golden Eagle website was fully rebuilt ~2 years ago; lost organic search performance and has not recovered to previous levels.

## Juan (Wang/Wong)
- Anna's father; introduced the consulting team to Golden Eagle.
- Has had prior calls with Jay and Tammy about AI in marketing and sales.
- At his own company, mandated Claude usage for all employees ~3 months ago (purchased licenses for everyone).
- Also instituted a rule that all meetings he or his team attend are recorded (transcript cut off mid-sentence before full detail

## 5. Assemble the full system prompt — eyeball it

In [7]:
SYSTEM_PREFIX = """You are an AI assistant embedded in Sundial, a consulting call management tool.
You help consultants understand their calls and project notes.
Be concise, factual, and grounded in the provided context.
If the answer isn't in the context, say so — don't speculate."""


def build_system_prompt(
    job_ids:  list[str] = [],
    file_rels: list[str] = [],
    include_transcripts: bool = True,
) -> str:
    """
    Assemble the full system prompt from selected calls and files.
    Returns (system_prompt_str, estimated_tokens).
    """
    blocks = [SYSTEM_PREFIX]

    if job_ids or file_rels:
        blocks.append('\n\n# Context')

    for job_id in job_ids:
        blocks.append('\n' + load_call_context(job_id, include_transcript=include_transcripts))

    for rel in file_rels:
        blocks.append('\n' + load_file_context(rel))

    system = '\n'.join(blocks)
    return system


# ── Preview the system prompt ─────────────────────────────────────────────────
system = build_system_prompt(
    job_ids   = [TEST_JOB_ID],
    file_rels = [TEST_FILE_REL] if TEST_FILE_REL else [],
)

est_tokens = len(system) // 4
print(f'System prompt: {len(system):,} chars  (~{est_tokens:,} tokens estimated)\n')
print('─' * 60)
print(system[:30000])
print(f'\n... ({len(system) - 30000:,} more chars)' if len(system) > 30000 else '')

System prompt: 41,993 chars  (~10,498 tokens estimated)

────────────────────────────────────────────────────────────
You are an AI assistant embedded in Sundial, a consulting call management tool.
You help consultants understand their calls and project notes.
Be concise, factual, and grounded in the provided context.
If the answer isn't in the context, say so — don't speculate.


# Context

## Call: 260403_Sundial build checkin (2026-04-03)

### Summary
This was an internal working session between Ana and Daniel. The call covered progress updates on their respective builds — Ana demonstrated her meeting intelligence dashboard (GUI, email template generation, and a pipeline that auto-updates Markdown knowledge files from call transcripts), and Daniel showed an annotation tool he built for collecting structured UI feedback. They discussed technical architecture questions including knowledge base design, Claude skill-based workflows, Obsidian as a comparator, and how to integrate the fou

## 6. Count tokens before calling

Use the Anthropic token counting API — exact count, no estimation.

In [8]:
TEST_QUESTION = 'Who was on this call and what were the main things discussed?'

token_response = client.messages.count_tokens(
    model    = MODEL,
    system   = system,
    messages = [{'role': 'user', 'content': TEST_QUESTION}],
)

input_tokens = token_response.input_tokens
print(f'Exact input token count: {input_tokens:,}')
print(f'Estimated was:           {est_tokens:,}  ({abs(input_tokens - est_tokens) / input_tokens * 100:.1f}% off)')
print()

# Context window is 200k for Sonnet. Budget check:
CONTEXT_WINDOW = 200_000
MAX_OUTPUT     = 1_024
available      = CONTEXT_WINDOW - input_tokens - MAX_OUTPUT
print(f'Context window:   {CONTEXT_WINDOW:>10,}')
print(f'Used (input):     {input_tokens:>10,}')
print(f'Reserved (output):{MAX_OUTPUT:>10,}')
print(f'Available buffer: {available:>10,}  ({available/CONTEXT_WINDOW*100:.1f}% remaining)')

if input_tokens > 150_000:
    print('\n⚠️  High token count — consider dropping transcripts and keeping briefings only')
else:
    print('\n✓ Well within budget')

Exact input token count: 10,020
Estimated was:           10,498  (4.8% off)

Context window:      200,000
Used (input):         10,020
Reserved (output):     1,024
Available buffer:    188,956  (94.5% remaining)

✓ Well within budget


## 7. First real call — single question, non-streaming

In [9]:
response = call_with_retry(
    model      = MODEL,
    max_tokens = MAX_OUTPUT,
    system     = system,
    messages   = [{'role': 'user', 'content': TEST_QUESTION}],
)

answer = response.content[0].text
print(f'Q: {TEST_QUESTION}\n')
print(f'A: {answer}\n')
print('─' * 40)
print(f'Input tokens:  {response.usage.input_tokens:,}')
print(f'Output tokens: {response.usage.output_tokens:,}')
print(f'Stop reason:   {response.stop_reason}')

Q: Who was on this call and what were the main things discussed?

A: ## Call Participants

- **Ana** — Consulting team; CS/AI/marketing background, building the meeting intelligence dashboard
- **Daniel** — Consulting team; IBM IT implementation background, building CRM integration and phone system API investigation

---

## Main Topics Discussed

1. **Ana's dashboard demo** — She showed a working pipeline that auto-updates Markdown knowledge files from call transcripts, a GUI for querying across calls and individual contacts, and an auto-generated email follow-up template (minimal prompt engineering required).

2. **Knowledge base architecture** — Sparked by a Karpathy tweet, they discussed using LLMs for personal/organizational knowledge bases, with Obsidian and McKinsey's W3 intranet as comparators. Ana explained her implementation uses an index file with pointers to Markdown files, updated via a find-and-replace mechanism based on Claude's leaked internal approach.

3. **Daniel's a

## 8. Multi-turn conversation

In [10]:
def chat(messages: list[dict], question: str) -> tuple[str, list[dict]]:
    """Send a message, append to history, return (answer, updated_history)."""
    messages = messages + [{'role': 'user', 'content': question}]
    resp = call_with_retry(
        model      = MODEL,
        max_tokens = MAX_OUTPUT,
        system     = system,
        messages   = messages,
    )
    answer   = resp.content[0].text
    messages = messages + [{'role': 'assistant', 'content': answer}]
    print(f'User:      {question}')
    print(f'Assistant: {answer}')
    print(f'           [{resp.usage.input_tokens} in / {resp.usage.output_tokens} out tokens]\n')
    return answer, messages


history = []

_, history = chat(history, 'Who was on this call?')
_, history = chat(history, 'What action items came out of it?')
_, history = chat(history, 'Summarize those in one sentence.')

User:      Who was on this call?
Assistant: The call was between **Ana** and **Daniel**, both on the consulting team.

- **Ana** — CS/AI/marketing background, building the meeting intelligence dashboard
- **Daniel** — IBM IT implementation background, building CRM integration and investigating the phone system API
           [10013 in / 59 out tokens]

User:      What action items came out of it?
Assistant: Here are the action items from the call, organized by owner:

**Daniel:**
- Email Sean to clarify the Elevate phone system API and what they have
- Build CRM integration endpoints and auto-suggestion logic (e.g., one-click reminder prompts based on client communication history)
- Research and plan channel integration for the four communication inputs (call, video, text, email)
- Share the annotation tool/skill file with Ana

**Ana:**
- Add a proper backend to the annotation tool for use in client demo feedback
- Talk to Sean about data architecture options (e.g., Claude skill vs. GU

## 9. Multi-item context — 2 calls + a file

Can the model distinguish between calls and answer cross-call questions?

In [11]:
# Pick 2 calls with briefings
calls_with_briefing = [r for r in rows if r['briefing']]

if len(calls_with_briefing) < 2:
    print('Need at least 2 calls with briefings — run the worker first')
else:
    job_id_1 = calls_with_briefing[0]['id']
    job_id_2 = calls_with_briefing[1]['id']
    name_1   = calls_with_briefing[0]['transcript_name']
    name_2   = calls_with_briefing[1]['transcript_name']

    multi_system = build_system_prompt(
        job_ids    = [job_id_1, job_id_2],
        file_rels  = [TEST_FILE_REL] if TEST_FILE_REL else [],
        include_transcripts = True,
    )

    tc = client.messages.count_tokens(
        model=MODEL, system=multi_system,
        messages=[{'role': 'user', 'content': 'test'}],
    )
    print(f'Multi-context: {tc.input_tokens:,} tokens for [{name_1}] + [{name_2}]')
    if TEST_FILE_REL:
        print(f'  + file: {TEST_FILE_REL}\n')

    q = f'Compare the two calls — what was different about the topics discussed in "{name_1}" vs "{name_2}"?'
    resp = call_with_retry(
        model=MODEL, max_tokens=1024,
        system=multi_system,
        messages=[{'role': 'user', 'content': q}],
    )
    print(f'Q: {q}\n')
    print(f'A: {resp.content[0].text}')

Multi-context: 17,028 tokens for [260403_Sundial build checkin] + [260402_Jon Wendt]
  + file: other-notes.md

Q: Compare the two calls — what was different about the topics discussed in "260403_Sundial build checkin" vs "260402_Jon Wendt"?

A: The two calls were fundamentally different in purpose, participants, and content:

## Purpose
- **260403 (Sundial build checkin)** was an internal working session between Ana and Daniel — focused on their own build progress, technical architecture, and next steps for the product they're developing.
- **260402 (Jon Wendt)** was an external discovery interview — Ana and Daniel interviewing a Golden Eagle sales rep to understand his workflow and surface pain points.

## Topics Covered

| Theme | 260403 (Internal) | 260402 (Jon Wendt) |
|---|---|---|
| **Focus** | What Ana and Daniel are building | How John does his job |
| **Tech discussion** | Deep — knowledge base architecture, Claude integrations, Elevate API, CRM endpoints, front-end stack deci

## 10. Streaming

Tokens printed as they arrive. This is what gets wired to SSE in the API endpoint.

In [12]:
import sys

question = 'What are all the action items across everything in context? Be thorough.'

print(f'Q: {question}\nA: ', end='', flush=True)

full_text    = ''
input_toks   = 0
output_toks  = 0

with client.messages.stream(
    model      = MODEL,
    max_tokens = 1024,
    system     = system,
    messages   = [{'role': 'user', 'content': question}],
) as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)
        full_text += text

    final = stream.get_final_message()
    input_toks  = final.usage.input_tokens
    output_toks = final.usage.output_tokens

print(f'\n\n[{input_toks} in / {output_toks} out tokens]')

Q: What are all the action items across everything in context? Be thorough.
A: Here is a complete list of action items pulled from all sources in context:

---

## Daniel's Action Items

1. **Investigate the Elevate phone system API** — send an email to Sean to clarify what they have (call recording, SMS access, etc.)
2. **Build CRM integration endpoints and auto-suggestion logic** — e.g., one-click reminder prompts based on client communication history (e.g., "first call with this client, set a reminder for one week?")
3. **Research and plan channel integration** for all four communication inputs: calls (Elevate), video (Teams), SMS (Elevate), and email (possibly custom IMAP)
4. **Share the annotation tool / skill file with Ana**

---

## Ana's Action Items

5. **Add a proper backend to the annotation tool** so it can be used when sending demos to clients for feedback collection
6. **Talk to Sean about data architecture options** (e.g., Claude skill vs. custom GUI approach) — do not p

## 11. Token budget stress test — all calls

Load every call with a briefing. Measure total tokens.
Identify where we hit the budget threshold and test the fallback (briefing only, no transcript).

In [13]:
BUDGET_WARNING  = 60_000   # above this: consider dropping transcripts
BUDGET_HARD_CAP = 150_000  # above this: definitely drop transcripts

print('Building context for each call individually:\n')
print(f'{"Name":<42} {"chars":>7}  {"est tok":>8}  notes')
print('─' * 72)

cumulative_chars = len(SYSTEM_PREFIX)

for r in calls_with_briefing:
    ctx_block = load_call_context(r['id'], include_transcript=True)
    ctx_chars = len(ctx_block)
    est       = ctx_chars // 4
    cumulative_chars += ctx_chars
    cum_est   = cumulative_chars // 4
    flag      = '⚠️ approaching limit' if cum_est > BUDGET_WARNING else ''
    flag      = '🛑 drop transcripts'  if cum_est > BUDGET_HARD_CAP else flag
    print(f"{r['transcript_name']:<42} {ctx_chars:>7,}  {est:>8,}  cumulative: {cum_est:,} {flag}")

print()
print(f'Total (all calls, with transcripts): ~{cumulative_chars // 4:,} tokens')

# Now count same but briefings only (no transcripts)
brief_chars = len(SYSTEM_PREFIX)
for r in calls_with_briefing:
    brief_chars += len(load_call_context(r['id'], include_transcript=False))
print(f'Total (all calls, briefings only):   ~{brief_chars // 4:,} tokens')

Building context for each call individually:

Name                                         chars   est tok  notes
────────────────────────────────────────────────────────────────────────
260403_Sundial build checkin                32,444     8,111  cumulative: 8,176 
260402_Jon Wendt                            29,021     7,255  cumulative: 15,432 
260402_Justin Jankowski                     38,067     9,516  cumulative: 24,948 
260402_Jodi Parmeter                        33,840     8,460  cumulative: 33,408 
260401_Jeff Parmeter                        32,056     8,014  cumulative: 41,422 
260401_Lukas Eichinger                      31,036     7,759  cumulative: 49,181 
260401_Shon Flaherty                        37,115     9,278  cumulative: 58,460 
call-05-followup                            18,292     4,573  cumulative: 63,033 ⚠️ approaching limit
call-04-crm-frontend-design                 23,195     5,798  cumulative: 68,832 ⚠️ approaching limit
call-03-sales-crm-followup          

In [14]:
# ── Exact count via API for the full corpus ───────────────────────────────────
all_job_ids = [r['id'] for r in calls_with_briefing]

# With transcripts
full_system = build_system_prompt(job_ids=all_job_ids, include_transcripts=True)
tc_full = client.messages.count_tokens(
    model=MODEL, system=full_system,
    messages=[{'role': 'user', 'content': 'test'}],
)

# Briefings only
brief_system = build_system_prompt(job_ids=all_job_ids, include_transcripts=False)
tc_brief = client.messages.count_tokens(
    model=MODEL, system=brief_system,
    messages=[{'role': 'user', 'content': 'test'}],
)

print(f'All {len(all_job_ids)} calls — with transcripts:  {tc_full.input_tokens:>8,} tokens')
print(f'All {len(all_job_ids)} calls — briefings only:    {tc_brief.input_tokens:>8,} tokens')
print(f'Savings from dropping transcripts:    {tc_full.input_tokens - tc_brief.input_tokens:>8,} tokens')
print(f'Context window remaining (briefings): {200_000 - tc_brief.input_tokens:>8,} tokens')

All 13 calls — with transcripts:   102,227 tokens
All 13 calls — briefings only:      19,223 tokens
Savings from dropping transcripts:      83,004 tokens
Context window remaining (briefings):  180,777 tokens


## Summary / Takeaways

Fill in after running:

- **Single call token count**: ~`???` tokens (with transcript)
- **All calls, full**: ~`???` tokens — fits / doesn't fit in 200k window
- **All calls, briefings only**: ~`???` tokens — fits comfortably
- **Truncation threshold**: drop transcripts when selecting > `?` calls simultaneously
- **Response quality**: how well does the model ground in context?
- **Streaming latency**: first token arrives in ~`?`s

These numbers inform the token budget logic in `/api/chat`.